In [ ]:
# =============================================================================
# Fig3: Decomposition of mortality change — Population / Ageing / PM2.5
# Layout: 2 rows × 3 cols
#   Row 0 (2/3 height): waterfall panels — 2020-2040 | 2040-2060 | 2020-2060
#   Row 1 (1/3 height): distribution panels — pop effect | age effect | rate effect
#
# East/West: Hu Huanyong line (胡焕庸线), same as Fig1
# HS group:  from Excel sheet earlypeak_NZ_CL
# Decomposition: Symmetric Path Average (exact, zero residual)
# =============================================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import geopandas as gpd
from matplotlib.gridspec import GridSpec
from pyproj import Transformer
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# ── 1. Paths ──────────────────────────────────────────────────────────────────

CSV_2020  = Path("/Volumes/UCL/论文工作/空气污染/health_burden/air_scenarios_5/earlypeak_NZ_CL_2020.csv")
CSV_2040  = Path("/Volumes/UCL/论文工作/空气污染/health_burden/air_scenarios_5/earlypeak_NZ_CL_2040.csv")
CSV_2060  = Path("/Volumes/UCL/论文工作/空气污染/health_burden/air_scenarios_5/earlypeak_NZ_CL_2060.csv")
XLSX_PATH = Path("/Volumes/UCL/论文工作/空气污染/清华城市健康指数-健康服务.xlsx")
SHP_PATH  = Path("/Users/shirley/Library/CloudStorage/OneDrive-UniversityCollegeLondon/Desktop/city_shp/shi_en.shp")
OUTFILE   = Path("/Users/shirley/Desktop/plots_V2/Fig4.png")

W_IN = 18 / 2.54
H_IN = 13 / 2.54 
DPI  = 400

# ── 2. HS group config ────────────────────────────────────────────────────────

HS_LEVELS = ["Poor", "Moderate", "Intermediate", "Good", "Excellent"]
COL_GROUP = {
    "Poor":         "#d7191c",
    "Moderate":     "#f47920",
    "Intermediate": "#f6c101",
    "Good":         "#abd9e9",
    "Excellent":    "#2c7bb6",
}

COL_AGE  = "#ca0020"
COL_POP  = "#4db3b3"
COL_RISK = "#bababa"

# ── 3. Read raw data ──────────────────────────────────────────────────────────

df20 = pd.read_csv(CSV_2020)
df40 = pd.read_csv(CSV_2040)
df60 = pd.read_csv(CSV_2060)

# ── 4. HS classification from Excel ──────────────────────────────────────────

xlsx_df = pd.read_excel(XLSX_PATH, sheet_name="earlypeak_NZ_CL")
hs_map = (
    xlsx_df[["city", "HS_type"]]
    .drop_duplicates(subset="city")
    .dropna(subset=["HS_type"])
    .query("HS_type != 0")
    .rename(columns={"HS_type": "HS_type"})
)

# ── 5. East / West by Hu Huanyong line（经纬度直接判别）──────────────────

# 胡焕庸线两端点（经度, 纬度）
# 北端：黑河  南端：腾冲
HHY = {
    "lon": [127.5, 98.5],
    "lat": [50.2,  25.0],
}

def hhy_lon_at_lat(lat):
    """给定纬度，返回胡焕庸线在该纬度的经度"""
    t = (lat - HHY["lat"][1]) / (HHY["lat"][0] - HHY["lat"][1])
    return HHY["lon"][1] + t * (HHY["lon"][0] - HHY["lon"][1])

# 用 WGS84 坐标读取城市重心
city_shp_geo = gpd.read_file(SHP_PATH)   # 保持 EPSG:4326，不投影
cx = city_shp_geo.geometry.centroid.x    # 经度
cy = city_shp_geo.geometry.centroid.y    # 纬度

city_shp_geo["region"] = np.where(
    cx > hhy_lon_at_lat(cy), "East", "West"
)

region_map = (
    city_shp_geo[["English", "region"]]
    .rename(columns={"English": "city"})
)

# ── 6. Decomposition ──────────────────────────────────────────────────────────

def decompose_all(df_t0, df_t1, period):
    results = []
    for city in df_t0["city"].unique():
        d0 = (df_t0[df_t0["city"] == city].groupby("age").agg(pop=("pop", "sum"), D=("mo_total", "sum")).reset_index())
        d1 = (df_t1[df_t1["city"] == city].groupby("age").agg(pop=("pop", "sum"), D=("mo_total", "sum")).reset_index())
        d0["r"], d1["r"] = d0["D"]/d0["pop"], d1["D"]/d1["pop"]
        N0, N1 = d0["pop"].sum(), d1["pop"].sum()
        w0, w1 = d0["pop"].values / N0, d1["pop"].values / N1
        r0, r1 = d0["r"].values, d1["r"].values
        D0, D1 = N0 * (w0 * r0).sum(), N1 * (w1 * r1).sum()
        p1_pop, p1_age, p1_risk = N1*(w0*r0).sum()-D0, N1*(w1*r0).sum()-N1*(w0*r0).sum(), N1*(w1*r1).sum()-N1*(w1*r0).sum()
        p2_risk, p2_age, p2_pop = N0*(w0*r1).sum()-D0, N0*(w1*r1).sum()-N0*(w0*r1).sum(), N1*(w1*r1).sum()-N0*(w1*r1).sum()
        results.append({
            "city": city, "period": period, "delta": D1 - D0,
            "pop_effect": (p1_pop + p2_pop)/2, "age_effect": (p1_age + p2_age)/2, "risk_effect": (p1_risk + p2_risk)/2,
        })
    return pd.DataFrame(results)

decomp = pd.concat([decompose_all(df20, df40, "2020–2040"), decompose_all(df40, df60, "2040–2060"), decompose_all(df20, df60, "2020–2060")], ignore_index=True)
decomp = decomp.merge(hs_map, on="city", how="inner").merge(region_map, on="city", how="left")
decomp["HS_type"] = pd.Categorical(decomp["HS_type"], categories=HS_LEVELS, ordered=True)

# ── 7. Global style ───────────────────────────────────────────────────────────

plt.rcParams.update({"font.family": "Arial", "font.size": 6, "axes.labelsize": 7, "axes.titlesize": 6, "axes.titleweight": "bold", "xtick.labelsize": 5, "ytick.labelsize": 5})
N_HS = len(HS_LEVELS)

# ── 8. Waterfall panel function (Left-Top Inset & K ticks) ────────────────────

def draw_waterfall(ax, period, tag):
    data = decomp[decomp["period"] == period]

    # --- Main Waterfall ---
    for hi, hs in enumerate(HS_LEVELS):
        for region, x_off, alpha in [("East", -0.22, 0.92), ("West", 0.22, 0.50)]:
            sub = data[(data["HS_type"] == hs) & (data["region"] == region)]
            if len(sub) == 0: continue
            pop_m, age_m, rate_m, net_m = sub["pop_effect"].mean(), sub["age_effect"].mean(), sub["risk_effect"].mean(), sub["delta"].mean()
            x, bw = hi + x_off, 0.19

            pos_btm = 0.0
            for val, col in [(age_m, COL_AGE), (max(pop_m, 0), COL_POP)]:
                if val > 1e-9:
                    ax.bar(x, val, bottom=pos_btm, width=bw, color=col, alpha=alpha, lw=0, zorder=3)
                    pos_btm += val
            neg_btm = 0.0
            for val, col in [(rate_m, COL_RISK), (min(pop_m, 0), COL_POP)]:
                if val < -1e-9:
                    ax.bar(x, val, bottom=neg_btm, width=bw, color=col, alpha=alpha, lw=0, zorder=3)
                    neg_btm += val
            marker = "D" if region == "East" else "o"
            ax.plot(x, net_m, marker=marker, ms=2, color="black", alpha=alpha, zorder=5, lw=0)

    # --- Inset Plot (National Total) ---
    # loc="upper left", 设置在左上角
    ax_ins = inset_axes(ax, width="10%", height="30%", loc="upper left",bbox_to_anchor=(0.12, 0.5, 0.5, 0.5),bbox_transform=ax.transAxes, borderpad=1.5)
    t_pop, t_age, t_risk = data["pop_effect"].sum(), data["age_effect"].sum(), data["risk_effect"].sum()
    
    ibw = 0.2 # 减小柱体宽度
    ax_ins.bar(0, t_age, width=ibw, color=COL_AGE, lw=0)
    ax_ins.bar(0, t_risk, width=ibw, color=COL_RISK, lw=0)
    p_btm = t_age if t_pop > 0 else t_risk
    ax_ins.bar(0, t_pop, bottom=p_btm, width=ibw, color=COL_POP, lw=0)
    
    ax_ins.axhline(0, color="black", lw=0.3)
    ax_ins.set_xticks([]); ax_ins.set_title("All cities", fontsize=4.5, pad=2)
    ax_ins.spines[['top', 'right']].set_visible(False)
    ax_ins.spines[["bottom", "left"]].set_linewidth(0.4)
    ax_ins.tick_params(width=0.4)
    # 设置 y 轴刻度为 K (例如 100000 -> 100K)
    ax_ins.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, pos: f'{x/1000:g}K'))
    ax_ins.tick_params(labelsize=4, length=2, pad=1)

    # --- Main Axis Formatting ---
    ax.axhline(0, color="grey", lw=0.3, ls="--", zorder=1)
    ax.set_xticks(range(N_HS))
    ax.set_xticklabels(HS_LEVELS, fontsize=5, rotation=15, ha="center")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, pos: f'{x/1000:g}K'))
    ax.set_ylabel("Change in deaths", fontsize=6)
    ax.spines[["top", "right"]].set_visible(False)
    ax.text(-0.12, 1.03, tag, transform=ax.transAxes, fontsize=8, fontweight="bold", va="bottom")
    ax.set_xlim(-0.6, N_HS - 0.4)
    ax.spines[["bottom", "left"]].set_linewidth(0.4)
    ax.tick_params(width=0.4)



# ── 9. Distribution panel (带端点横线和 K 刻度) ──────────────────────────────────

def draw_dist_panel(ax, factor, ylabel, tag):
    rng = np.random.default_rng(42)
    data = decomp[decomp["period"] == "2020–2060"]
    for hi, hs in enumerate(HS_LEVELS):
        color = COL_GROUP[hs]
        for region, x_off, solid in [("East", -0.18, True), ("West", 0.18, False)]:
            sub = data[(data["HS_type"] == hs) & (data["region"] == region)][factor].dropna()
            sub = sub[np.isfinite(sub)]
            if len(sub) < 2: continue
            
            x, bw, ls, alp = hi + x_off, 0.07, ("-" if solid else "--"), (1.0 if solid else 0.65)
            q1, med, q3 = np.percentile(sub, [25, 50, 75])
            iqr = q3 - q1
            lo, hi_ = sub[sub >= q1 - 1.5 * iqr].min(), sub[sub <= q3 + 1.5 * iqr].max()
            
            # 1. 散点
            ax.scatter(x + rng.uniform(-0.05, 0.05, len(sub)), sub, color=color, s=1, alpha=0.18, lw=0, zorder=2)
            
            # 2. 箱体
            rect = mpatches.FancyBboxPatch((x-bw, q1), 2*bw, q3-q1, boxstyle="square,pad=0", 
                                           facecolor="white", edgecolor=color, lw=0.7, ls=ls, alpha=alp, zorder=3)
            ax.add_patch(rect)
            
            # 3. 中位数线
            ax.plot([x-bw, x+bw], [med, med], color=color, lw=1.0, ls=ls, zorder=4)
            
            # 4. 须线 (垂直)
            ax.plot([x, x], [lo, q1], color=color, lw=0.6, ls=ls, zorder=3)
            ax.plot([x, x], [q3, hi_], color=color, lw=0.6, ls=ls, zorder=3)
            
            # --- 新增：范围两端的短横线 (Caps) ---
            cap_w = 0.03  # 横线的宽度
            ax.plot([x - cap_w, x + cap_w], [lo, lo], color=color, lw=0.6, ls=ls, zorder=3)
            ax.plot([x - cap_w, x + cap_w], [hi_, hi_], color=color, lw=0.6, ls=ls, zorder=3)
            
    # 坐标轴修饰
    ax.axhline(0, color="grey", lw=0.3, ls="--", zorder=1)
    ax.set_xticks(range(N_HS))
    ax.set_xticklabels(HS_LEVELS, fontsize=5, rotation=15, ha="center")
    ax.set_ylabel(ylabel, fontsize=6)
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["bottom", "left"]].set_linewidth(0.4)
    ax.tick_params(width=0.4) # 减小刻度线
    
    # 设置 y 轴为 K 刻度
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, pos: f'{x/1000:g}K'))
    
    ax.text(-0.12, 1.03, tag, transform=ax.transAxes, fontsize=8, fontweight="bold", va="bottom")

# ── 10. Build figure ──────────────────────────────────────────────────────────

fig = plt.figure(figsize=(W_IN, H_IN), dpi=DPI)
gs = GridSpec(nrows=2, ncols=3, figure=fig, height_ratios=[1.5, 1], hspace=0.25, wspace=0.35)

ax_wf_0 = fig.add_subplot(gs[0, 0])
ax_wf_1 = fig.add_subplot(gs[0, 1], sharey=ax_wf_0)
ax_wf_2 = fig.add_subplot(gs[0, 2], sharey=ax_wf_0)
ax_wf = [ax_wf_0, ax_wf_1, ax_wf_2]
ax_ds = [fig.add_subplot(gs[1, c]) for c in range(3)]

PERIODS, TAGS_TOP, TAGS_BOT = ["2020–2040", "2040–2060", "2020–2060"], ["a", "b", "c"], ["d", "e", "f"]
FACTORS = [("pop_effect", "Population size effect"), ("age_effect", "Ageing effect"), ("risk_effect", "PM$_{2.5}$ risk effect")]

for i, (ax, period, tag) in enumerate(zip(ax_wf, PERIODS, TAGS_TOP)):
    draw_waterfall(ax, period, tag)
    ax.set_title(period, fontsize=6, pad=3)

for ax, (factor, ylabel), tag in zip(ax_ds, FACTORS, TAGS_BOT):
    draw_dist_panel(ax, factor, ylabel, tag)

# ── 11. Shared legend ─────────────────────────────────────────────────────────

# 1. 因子 handles (Ageing, Pop, Rate)
factor_handles = [
    mpatches.Patch(color=COL_AGE,  label="Ageing"),
    mpatches.Patch(color=COL_POP,  label="Population size"),
    mpatches.Patch(color=COL_RISK, label="PM$_{2.5}$ risk"),
]

# 2. 区域标识 handles (East/West 的形状与线型)
ew_handles = [
    # 线型标识 (关键补充)
    plt.Line2D([0], [0], marker="D", color="black", ms=2, ls="none", label="East (net)"),
    plt.Line2D([0], [0], marker="o", color="black", ms=2, ls="none", alpha=0.5, label="West (net)"),
]

# 3. 柱子透明度标识 handles
bar_handles = [
    mpatches.Patch(facecolor="grey", alpha=0.9, label="East (bar)"),
    mpatches.Patch(facecolor="grey", alpha=0.45, label="West (bar)"),
]

li_handles = [   
    plt.Line2D([0], [0], color="grey", lw=1, ls="-",  label="East (solid)"),
    plt.Line2D([0], [0], color="grey", lw=1, ls="--", label="West (dashed)")]
all_handles = factor_handles + bar_handles+ew_handles + li_handles

fig.legend(
    handles=all_handles,
    loc="lower center",
    ncol=10,          # 增加列数或调整为 2 行，防止太拥挤
    fontsize=5.5,
    frameon=False,
    bbox_to_anchor=(0.5, 0), # 略微往下挪，避免与 x 轴标签冲突
    handlelength=1.5,
    columnspacing=1.0,
)
# ── 12. Save ──────────────────────────────────────────────────────────────────

OUTFILE.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(OUTFILE, dpi=DPI, bbox_inches="tight", facecolor="white")
plt.close(fig)
print(f"✓ Saved → {OUTFILE}")

/var/folders/xt/zxpvbvsx2zb75gd2395zm9000000gn/T/ipykernel_44221/301885658.py:84: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  cx = city_shp_geo.geometry.centroid.x    # 经度
/var/folders/xt/zxpvbvsx2zb75gd2395zm9000000gn/T/ipykernel_44221/301885658.py:85: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  cy = city_shp_geo.geometry.centroid.y    # 纬度


✓ Saved → /Users/shirley/Desktop/plots_V2/Fig4.png
